# 🧑‍💼 Supervisor Architecture in LangGraph## Learning ObjectivesIn this notebook, you will learn:1. **Supervisor Pattern** - How one "manager" agent coordinates multiple specialist agents instead of a single agent doing everything2. **Structured Routing** - How to use `with_structured_output` so the supervisor's next-step decision is a validated Pydantic object, not free text3. **Shared Graph State** - How `SupervisorState` threads messages and routing metadata between the supervisor and specialist nodes4. **Conditional Edges** - How `add_conditional_edges` lets the graph loop back to the supervisor after every specialist call until the task is marked complete## Prerequisites- Familiarity with core LangGraph concepts (`StateGraph`, nodes, edges) from `03_LangGraph_Fundamentals`- Basic understanding of Pydantic models for structured output- An `OPENAI_API_KEY` in a `.env` file at the project root

---## 🔧 Part 1: Environment SetupWe load environment variables and import everything the notebook needs: LangGraph's graph-building primitives, LangChain's chat model and message types, and Pydantic for the supervisor's structured routing decision.

In [ ]:
# ============================================================================# ENVIRONMENT SETUP: Imports and Configuration# ============================================================================from typing import Literalfrom typing_extensions import Annotated, TypedDictfrom dotenv import load_dotenvfrom pydantic import BaseModel, Fieldfrom langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessagefrom langchain_core.prompts import ChatPromptTemplatefrom langchain_openai import ChatOpenAIfrom langgraph.graph import END, START, StateGraphfrom langgraph.graph.message import add_messages# Loads OPENAI_API_KEY (and any other secrets) from a local .env fileload_dotenv()print("✅ Environment loaded and imports ready!")

---## 🗂️ Part 2: Define the Shared Graph StateEvery node in the graph reads from and writes to a single shared state object. `SupervisorState` tracks the running message history plus two pieces of routing metadata: which agent should act next, and whether the task is finished.### Key Concepts:- **`messages`**: Uses `add_messages` as a reducer so new messages are appended rather than overwriting the list- **`next_agent`**: Set by the supervisor node to tell the graph which specialist to route to- **`task_complete`**: A flag the supervisor sets once it decides the work is done

In [ ]:
# ============================================================================# SUPERVISOR STATE: Shared State Schema for the Graph# ============================================================================class SupervisorState(TypedDict):    messages: Annotated[list[BaseMessage], add_messages]    next_agent: str    task_complete: bool    final_response: str

### 🤖 Initialize the LLMA single `ChatOpenAI` instance is shared by the supervisor (for routing decisions) and every specialist node (for generating content). `temperature=0` keeps routing decisions deterministic.

In [ ]:
# ============================================================================# LLM INITIALIZATION: Shared Chat Model# ============================================================================llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)print(f"🤖 LLM initialized: {llm.model_name}")

---## 🏗️ Part 3: Build the Supervisor System`create_supervisor_system()` assembles the full graph: a **supervisor** node that decides which specialist should act next, three **specialist** nodes (researcher, writer, critic) that do the actual work, and a **finalize** node that extracts the final answer once the supervisor signals completion.### Key Insight:> The supervisor never does the work itself — it only routes. Each specialist runs, appends its output to `messages`, and control returns to the supervisor, which decides the *next* step. This loop continues until the supervisor responds with `FINISH`.

In [ ]:
# ============================================================================# SUPERVISOR SYSTEM: Build the Graph with a Router + Specialist Agents# ============================================================================def create_supervisor_system():    """Create a supervisor with specialist agents."""    # Define the routing schema    class RouteDecision(BaseModel):        next: Literal["researcher", "writer", "critic", "FINISH"] = Field(            description="The next agent to call, or FINISH if task is complete"        )        reasoning: str = Field(description="Why this agent was chosen")    supervisor_llm = llm.with_structured_output(RouteDecision)    # Supervisor node    def supervisor(state: SupervisorState) -> dict:        system_prompt = """You are a supervisor managing a team of specialists:        1. researcher - Gathers information and facts        2. writer - Creates content and text        3. critic - Reviews and improves work        Based on the conversation, decide which agent should act next.        If the task is complete, respond with FINISH.        Current conversation shows the progress so far."""        messages = [SystemMessage(content=system_prompt)] + state["messages"]        decision = supervisor_llm.invoke(messages)        if decision.next == "FINISH":            return {"next_agent": "FINISH", "task_complete": True}        return {            "next_agent": decision.next,            "messages": [                AIMessage(                    content=f"[Supervisor] Routing to {decision.next}: {decision.reasoning}"                )            ],        }    # Define specialist agents (for demo purposes, they just echo the task)    def researcher(state: SupervisorState) -> dict:        prompt = ChatPromptTemplate.from_messages(            [                (                    "system",                    "You are a research specialist. Gather facts and information relevant to the task. Be thorough but concise.",                ),                (                    "human",                    "Task context:\n{context}\n\nProvide your research findings.",                ),            ]        )        # Get task from first human message        task = next(            (m.content for m in state["messages"] if isinstance(m, HumanMessage)), ""        )        response = llm.invoke(prompt.format_messages(context=task))        return {"messages": [AIMessage(content=f"[Researcher] {response.content}")]}    def writer(state: SupervisorState) -> dict:        prompt = ChatPromptTemplate.from_messages(            [                (                    "system",                    "You are a writing specialist. Create clear, engaging content based on the available information.",                ),                ("human", "Previous work:\n{context}\n\nWrite the content."),            ]        )        context = "\n".join([m.content for m in state["messages"][-5:]])        response = llm.invoke(prompt.format_messages(context=context))        return {"messages": [AIMessage(content=f"[Writer] {response.content}")]}    def critic(state: SupervisorState) -> dict:        prompt = ChatPromptTemplate.from_messages(            [                (                    "system",                    "You are a quality critic. Review the work and provide constructive feedback. If the work is good, say so.",                ),                ("human", "Work to review:\n{context}\n\nProvide your critique."),            ]        )        context = "\n".join([m.content for m in state["messages"][-3:]])        response = llm.invoke(prompt.format_messages(context=context))        return {"messages": [AIMessage(content=f"[Critic] {response.content}")]}    def finalize(state: SupervisorState) -> dict:        # Get the last substantial response        for msg in reversed(state["messages"]):            if isinstance(msg, AIMessage) and "[Writer]" in msg.content:                content = msg.content.replace("[Writer] ", "")                return {"final_response": content}        return {"final_response": "Task completed."}    # Route based on supervisor decision    def route_to_agent(state: SupervisorState) -> str:        if state.get("task_complete"):            return "finalize"        return state["next_agent"]    graph = StateGraph(SupervisorState)    graph.add_node("supervisor", supervisor)    graph.add_node("researcher", researcher)    graph.add_node("writer", writer)    graph.add_node("critic", critic)    graph.add_node("finalize", finalize)    graph.add_edge(START, "supervisor")    graph.add_conditional_edges(        "supervisor",        route_to_agent,        {            "researcher": "researcher",            "writer": "writer",            "critic": "critic",            "finalize": "finalize",        },    )    # After each specialist, go back to supervisor    graph.add_edge("researcher", "supervisor")    graph.add_edge("writer", "supervisor")    graph.add_edge("critic", "supervisor")    graph.add_edge("finalize", END)    return graph.compile()

---## ▶️ Part 4: Run the DemosTwo demo functions exercise the graph end to end: one prints the full agent conversation and final answer, the other isolates just the supervisor's routing decisions so you can see the delegation logic in action.### 4.1 📰 `demo_supervisor`Runs a blog-writing task through the full supervisor loop and prints each agent's contribution plus the final response.

In [ ]:
# ============================================================================# DEMO: Full Supervisor Conversation# ============================================================================def demo_supervisor():    """Demo the supervisor system."""    agent = create_supervisor_system()    print("Supervisor Agent Demo:\n")    result = agent.invoke(        {            "messages": [                HumanMessage(                    content="Write a short blog post about the benefits of AI in healthcare"                )            ],            "next_agent": "",            "task_complete": False,            "final_response": "",        }    )    print("Agent conversation:")    for msg in result["messages"]:        if isinstance(msg, AIMessage):            print(f"\n{msg.content[:200]}...")    print(f"\n\nFinal Response:\n{result['final_response']}")

### 4.2 🧭 `demo_supervisor_trace`Runs a tagline-writing task but only prints the supervisor's routing decisions — useful for seeing *why* each specialist was chosen.

In [ ]:
# ============================================================================# DEMO: Supervisor Decision Trace# ============================================================================def demo_supervisor_trace():    """Show supervisor decision-making."""    agent = create_supervisor_system()    print("\nSupervisor Decision Trace:\n")    result = agent.invoke(        {            "messages": [                HumanMessage(                    content="Create a marketing tagline for a new coffee brand"                )            ],            "next_agent": "",            "task_complete": False,            "final_response": "",        }    )    print("Routing decisions:")    for msg in result["messages"]:        if isinstance(msg, AIMessage) and "[Supervisor]" in msg.content:            print(f"  → {msg.content}")

### 4.3 🚀 ExecuteThe original `__main__` guard, kept as-is. Jupyter sets `__name__` to `"__main__"`, so this cell runs directly — uncomment a line to switch which demo runs.

In [ ]:
# ============================================================================# RUN: Execute the Demo# ============================================================================if __name__ == "__main__":    # demo_supervisor()    demo_supervisor_trace()

---## 📝 SummaryIn this notebook, we learned:### 1. The Supervisor Pattern- **Delegation over doing**: The supervisor node only decides *who* acts next — it never performs the work itself- **Structured routing**: `with_structured_output(RouteDecision)` forces the supervisor's decision into a validated `next` + `reasoning` object instead of parsing free text- **Loop-back edges**: Every specialist (`researcher`, `writer`, `critic`) routes back to the supervisor, so the graph keeps delegating until `FINISH` is decided### 2. Shared State Design- **`SupervisorState`**: Threads `messages` (via `add_messages`), `next_agent`, `task_complete`, and `final_response` across every node- **`finalize` node**: Extracts the last `[Writer]`-tagged message as the deliverable once the supervisor marks the task complete### Next Steps- Explore the multi-agent **swarm** pattern, where agents hand off directly to each other instead of through a central supervisor- Try adding a fourth specialist (e.g. a fact-checker) and see how the supervisor's routing prompt needs to change